# Vector Databases: ChromaDB & FAISS
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/06_GenAI_LLM_RAG/faiss_chroma_vector_db.ipynb)

Embeddings only become useful at scale when you can query them fast. Vector databases index millions of vectors and return approximate nearest neighbors in milliseconds.

**Chroma** = batteries-included document store. **FAISS** = Meta's bare-metal similarity search library.

In [ ]:
!pip install -q chromadb faiss-cpu sentence-transformers

## 1. Build a knowledge base

In [ ]:
docs = [
    "The Eiffel Tower is located in Paris, France.",
    "Python lists are mutable; tuples are immutable.",
    "Photosynthesis converts sunlight into chemical energy.",
    "Pandas DataFrames are two-dimensional labeled structures.",
    "Mount Everest sits on the border of Nepal and China.",
    "List comprehensions provide concise loop syntax in Python.",
]
metas = [{"topic": t} for t in ["geo", "python", "bio", "python", "geo", "python"]]

## 2. ChromaDB - documents in, answers out

In [ ]:
import chromadb

client = chromadb.Client()                      # in-memory; use PersistentClient(path=...) to keep data
col = client.create_collection("kb")

col.add(documents=docs, metadatas=metas, ids=[f"d{i}" for i in range(len(docs))])

res = col.query(query_texts=["What makes tuples different?"], n_results=2)
for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
    print(f"[{meta['topic']}] {doc}   (distance={dist:.3f})")

Chroma embedded AND stored automatically. Filtered queries work too: `query(..., where={'topic': 'python'})`.

## 3. FAISS - manual but blazing fast

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss

model = SentenceTransformer("all-MiniLM-L6-v2")
vecs = np.array(model.encode(docs), dtype="float32")
faiss.normalize_L2(vecs)                        # cosine == inner product after normalize

index = faiss.IndexFlatIP(vecs.shape[1])        # exact search; try IndexIVFFlat for millions
index.add(vecs)

q = np.array([model.encode("How do I write a compact for-loop in Python?")], dtype="float32")
faiss.normalize_L2(q)
D, I = index.search(q, k=2)
for score, idx in zip(D[0], I[0]):
    print(f"{score:.3f}  {docs[idx]}")

## Choosing an option
| | Chroma | FAISS | Managed (Pinecone/Weaviate/Qdrant) |
|---|---|---|---|
| Setup | trivial | low | service account |
| Metadata filtering | built-in | DIY | built-in |
| Scale | prototype -> mid | huge, single-node | huge, distributed |
| Persistence | PersistentClient | save/load index files | cloud |

RAG pipeline order: chunk docs -> embed -> upsert -> retrieve top-k at question time -> stuff context into prompt.